In [1]:
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [2]:
import subprocess
import time

subprocess.Popen(['ollama', 'serve'])
time.sleep(5)
print("Ollama started!")

Ollama started!


In [3]:
!ollama pull llama3.2:3b

In [4]:
import requests

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.2:3b",
        "prompt": "Say hello in one word.",
        "stream": False
    }
)
print(response.json()['response'])

Hello.


In [5]:
from google.colab import files
uploaded = files.upload()
import pandas as pd
df = pd.read_csv('cleaned_data_cache.csv')
print(df.shape)
print(df.columns.tolist())

Saving cleaned_data_cache.csv to cleaned_data_cache (2).csv
(4841, 6)
['description', 'medical_specialty', 'sample_name', 'transcription', 'keywords', 'cleaned_transcription']


In [6]:
import requests
import json

JUDGE_SYSTEM_PROMPT = """You are an expert medical information extraction evaluator.

Your task is to evaluate the quality of structured medical information extracted from clinical transcriptions.

You will receive:
1. Original medical transcription
2. Extracted structured data (JSON)

Evaluate THREE criteria for EACH field:
1) Correctness — extracted items actually appear in the text
2) Completeness — important items present in text are not missing
3) Hallucinations — extracted items NOT supported by text

Output ONLY valid JSON:
{
  "field_scores": {
    "symptoms": 0.85,
    "diagnoses": 0.90,
    "medications": 0.75,
    "procedures": 0.95
  },
  "overall_score": 0.89,
  "missing_items": {
    "symptoms": [],
    "diagnoses": [],
    "medications": [],
    "procedures": []
  },
  "hallucinations": {
    "symptoms": [],
    "diagnoses": [],
    "medications": [],
    "procedures": []
  },
  "explanation": "Brief explanation"
}"""


def extract_medical_entities(transcription_text):
    prompt = f"""You are a medical information extraction system.
Extract entities from the medical transcription below.
Return ONLY a valid JSON object.

JSON format:
{{
    "diagnoses": ["list of diagnoses"],
    "medications": ["list of medications"],
    "symptoms": ["list of symptoms"],
    "procedures": ["list of procedures"]
}}

Medical transcription:
{transcription_text[:1000]}

JSON:"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": "llama3.2:3b", "prompt": prompt, "stream": False}
    )
    raw = response.json()['response'].strip()
    try:
        start = raw.find('{')
        end = raw.rfind('}') + 1
        return json.loads(raw[start:end])
    except:
        return {"raw_response": raw}


def create_judge_prompt(transcription, extracted_json):
    return f"""Evaluate this medical information extraction.

ORIGINAL TRANSCRIPTION:
---
{transcription[:3000]}
---

EXTRACTED DATA:
---
{json.dumps(extracted_json, indent=2)}
---

Return ONLY valid JSON following the scoring schema exactly."""


def evaluate_extraction(transcription, extracted_json):
    judge_prompt = create_judge_prompt(transcription, extracted_json)
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3.2:3b",
            "prompt": JUDGE_SYSTEM_PROMPT + "\n\n" + judge_prompt,
            "stream": False
        }
    )
    raw = response.json()['response'].strip()
    try:
        start = raw.find('{')
        end = raw.rfind('}') + 1
        return json.loads(raw[start:end])
    except:
        return {"raw_response": raw}

print("All functions defined!")

All functions defined!


In [7]:
samples = df.sample(5, random_state=42)

results = []
for idx, row in samples.iterrows():
    print(f"\n{'='*50}")
    print(f"Specialty: {row['medical_specialty']}")
    print(f"{'='*50}")


    extracted = extract_medical_entities(row['transcription'])
    print("Extracted:")
    print(json.dumps(extracted, indent=2))

    evaluation = evaluate_extraction(row['transcription'], extracted)
    print("\nJudge Score:", evaluation.get('overall_score', 'N/A'))

    results.append({
        'specialty': row['medical_specialty'],
        'extracted': extracted,
        'evaluation': evaluation
    })

print("\n=== SUMMARY ===")
scores = [r['evaluation'].get('overall_score', 0) for r in results if isinstance(r['evaluation'].get('overall_score'), float)]
if scores:
    print(f"Average overall score: {sum(scores)/len(scores):.3f}")


Specialty: General Medicine
Extracted:
{
  "diagnoses": [
    "Hydrocarbon aspiration",
    "Aplastic crisis"
  ],
  "medications": [],
  "symptoms": [
    "Dyspnea",
    "Pleuritic chest pain",
    "Hemoptysis",
    "Nausea",
    "Vomiting"
  ],
  "procedures": []
}

Judge Score: 0.79

Specialty: Obstetrics / Gynecology
Extracted:
{
  "diagnoses": [
    "Recurrent dysplasia of vulva"
  ],
  "medications": [],
  "symptoms": [
    "several slightly raised and pigmented lesions",
    "areas of acetowhite epithelium"
  ],
  "procedures": [
    "Carbon dioxide laser photo-ablation"
  ]
}

Judge Score: 0.92

Specialty: Consult - History and Phy.
Extracted:
{
  "diagnoses": [
    "Normocephalic"
  ],
  "medications": [],
  "symptoms": [
    "Blood pressure XXX",
    "Pulse XXX",
    "Temperature XXX",
    "Respirations XXX"
  ],
  "procedures": []
}

Judge Score: 0.93

Specialty: Pain Management
Extracted:
{
  "diagnoses": [
    "Low back pain"
  ],
  "medications": [],
  "symptoms": [],
  